# TH - LiteLLM Supply Chain (TeamPCP)

Hunting for indicators of the malicious `litellm` v1.82.7/v1.82.8 packages across customer environments. The compromised package drops a `.pth` auto-exec payload that harvests creds, encrypts them, exfils to `models.litellm[.]cloud`, and installs a systemd C2 backdoor polling `checkmarx[.]zone`.

**Date:** 2026-03-28  
**Author:** Josh Strickland  
**Platform:** LimaCharlie EDR  
**Advisory:** [GHSA-5mg7-485q-xm76](https://github.com/advisories/GHSA-5mg7-485q-xm76)  
**Exposure window:** 2026-03-24 10:39 to ~16:00 UTC (~5.5 hrs)  
**Target OS:** Linux (persistence), cross-platform (.pth trigger)  

## Hypothesis

Any endpoint that installed `litellm` v1.82.7 or v1.82.8 from PyPI on March 24 is compromised. v1.82.7 only triggers on proxy import; v1.82.8 added a `.pth` file that fires on every Python startup. The payload harvests creds and exfils them, then installs a systemd C2 backdoor.

LiteLLM's CI ran unpinned Trivy which was already backdoored by TeamPCP -- that's how the PyPI publish token was stolen. Source repo was never modified.

Refs: [HuskyHacks](https://huskyhacks.io/posts/litellm-cred-stealer/), [Wiz](https://www.wiz.io/blog/threes-a-crowd-teampcp-trojanizes-litellm-in-continuation-of-campaign), [Datadog](https://securitylabs.datadoghq.com/articles/litellm-compromised-pypi-teampcp-supply-chain-campaign/), [Snyk](https://snyk.io/articles/poisoned-security-scanner-backdooring-litellm/)

## Methodology

Wide sweep first with behavioral + IOC queries across all orgs, then targeted deep dive on hit orgs only (openssl encryption pipeline, systemd persistence). Correlate per-host to see how many attack stages landed. Validated against a controlled detonation before sweeping production.

### Setup

In [ ]:
# Install requirements into the running kernel (no restart needed)
# Pin limacharlie to v4.x -- v5 removed Manager/Replay classes
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "limacharlie", "-y"], stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "limacharlie<5", "pandas", "python-dotenv", "--quiet", "--no-cache-dir"])
print("Requirements installed")

In [ ]:
import limacharlie
from limacharlie.Replay import Replay
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import json
import time
import os
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()
print("Imports loaded, .env processed")

### Credentials

In [ ]:
LC_UID = os.environ["LC_UID"]
LC_API_KEY = os.environ["LC_API_KEY"]

assert LC_UID, "LC_UID not found in .env"
assert LC_API_KEY, "LC_API_KEY not found in .env"
print(f"UID: {LC_UID[:8]}...")
print("Credentials loaded")

### Discover Organizations

In [ ]:
mgr = limacharlie.Manager(uid=LC_UID, secret_api_key=LC_API_KEY)
result = mgr.userAccessibleOrgs(with_names=True, limit=500)
orgs = dict(zip(result['orgs'], [result['names'].get(oid, 'Unknown') for oid in result['orgs']]))

print(f"Found {len(orgs)} organizations:")
for oid, name in sorted(orgs.items(), key=lambda x: x[1]):
    print(f"  {name}: {oid[:12]}...")

### Filter Orgs (optional)

In [ ]:
# --- Uncomment to exclude specific orgs ---
# EXCLUDE_ORGS = {"example1", "example2"}
# orgs = {oid: name for oid, name in orgs.items() if name not in EXCLUDE_ORGS}

# --- Uncomment to target only specific orgs ---
# INCLUDE_ONLY = {"customer-a", "customer-b"}
# orgs = {oid: name for oid, name in orgs.items() if name in INCLUDE_ONLY}

print(f"Querying {len(orgs)} org(s)")

### Shared Query Function + Tuning Parameters

In [ ]:
LIMIT_PER_ORG = 5000
LIMIT_EVENT_SCAN = 50000
MAX_WORKERS = 15

def query_single_org(oid, org_name, query_name, lcql_query):
    """Run one LCQL query against one org. Returns (events_list, error_or_None)."""
    try:
        mgr = limacharlie.Manager(oid=oid, uid=LC_UID, secret_api_key=LC_API_KEY)
        replay = Replay(mgr)

        response = replay._doQuery(
            lcql_query,
            limitEvent=LIMIT_EVENT_SCAN,
            limitEval=None,
            isCursorBased=False,
            stream='event'
        )

        error = response.get('error', '')
        results = response.get('results', [])

        if error and not results and 'limit' not in error.lower():
            return [], f"Error: {error}"

        events = []
        for result in results:
            evt = result.get('data', result) if isinstance(result, dict) else None
            if not evt:
                continue

            routing = evt.get('routing', {})
            hostname = routing.get('hostname', '')

            if hostname == 'ext-cloud-cli':
                continue

            ts = routing.get('event_time', 0)
            if ts > 9999999999:
                ts = ts / 1000

            row = {
                'query_name': query_name,
                'org_name': org_name,
                'oid': oid,
                'timestamp': datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if ts else '',
                'event_type': routing.get('event_type', ''),
                'hostname': hostname,
                'sensor_id': routing.get('sid', ''),
                'event_data': json.dumps(evt.get('event', {}), default=str),
            }
            events.append(row)

        events.sort(key=lambda e: e['timestamp'], reverse=True)
        return events[:LIMIT_PER_ORG], None

    except Exception as e:
        return [], str(e)


def run_queries(queries, target_orgs, label=""):
    """Run a dict of {name: lcql} queries across target_orgs. Returns (all_events, errors)."""
    all_events = []
    errors = []
    start = time.time()

    total = len(queries) * len(target_orgs)
    print(f"{label}: {len(queries)} queries x {len(target_orgs)} orgs = {total} jobs")

    for query_name, lcql_query in queries.items():
        print(f"\n  --- {query_name} ---")

        workers = min(MAX_WORKERS, len(target_orgs))
        query_events = []

        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(query_single_org, oid, name, query_name, lcql_query): (oid, name)
                for oid, name in target_orgs.items()
            }

            for future in as_completed(futures):
                oid, name = futures[future]
                events, error = future.result()

                if error:
                    errors.append({'query': query_name, 'org': name, 'oid': oid, 'error': error})
                elif events:
                    query_events.extend(events)
                    print(f"      HIT: {name} -- {len(events)} events")

        if not query_events:
            print(f"      No hits")
        all_events.extend(query_events)

    elapsed = time.time() - start
    print(f"\n  {label} complete: {len(all_events)} events, {len(errors)} errors, {elapsed:.1f}s")
    return all_events, errors

print("Query function ready")

### Phase 1 - Wide Sweep

Behavioral query for the `.pth` base64 exec pattern + IOC query for attacker DNS across all orgs and platforms.

In [ ]:
PHASE1_QUERIES = {
    # Behavioral: .pth supply chain trigger -- python exec'ing base64 blob
    # All platforms -- .pth fires on Linux, Mac, and Windows
    "behavior_pth_b64_exec": (
        "-336h | * | NEW_PROCESS | "
        "event/FILE_PATH contains 'python' "
        "and event/COMMAND_LINE contains 'b64decode' "
        "and event/COMMAND_LINE contains 'exec'"
    ),

    # IOC: DNS to known TeamPCP infrastructure
    "ioc_attacker_dns": (
        "-336h | * | DNS_REQUEST | "
        "event/DOMAIN_NAME contains 'litellm.cloud' "
        "or event/DOMAIN_NAME contains 'checkmarx.zone'"
    ),
}

phase1_events, phase1_errors = run_queries(PHASE1_QUERIES, orgs, label="PHASE 1: WIDE SWEEP")

### Phase 1 Results

Pull out which orgs and hosts had hits. These become the targets for Phase 2.

In [ ]:
df_phase1 = pd.DataFrame(phase1_events)

if not df_phase1.empty:
    df_phase1['timestamp'] = pd.to_datetime(df_phase1['timestamp'], utc=True)
    df_phase1 = df_phase1.sort_values('timestamp', ascending=False).reset_index(drop=True)

    # Extract hit orgs and hosts
    hit_oids = set(df_phase1['oid'].unique())
    hit_orgs = {oid: name for oid, name in orgs.items() if oid in hit_oids}
    hit_hosts = set(df_phase1['hostname'].unique())

    print(f"PHASE 1 HITS: {len(df_phase1)} events across {len(hit_orgs)} org(s), {len(hit_hosts)} host(s)")
    print()
    print("Hit breakdown by query:")
    print(df_phase1['query_name'].value_counts().to_string())
    print()
    print("Hit orgs:")
    for oid, name in hit_orgs.items():
        count = len(df_phase1[df_phase1['oid'] == oid])
        hosts = df_phase1[df_phase1['oid'] == oid]['hostname'].nunique()
        print(f"  {name}: {count} events, {hosts} host(s)")
    print()
    print("Hit hosts:")
    for host in sorted(hit_hosts):
        queries = df_phase1[df_phase1['hostname'] == host]['query_name'].unique()
        print(f"  {host}: {', '.join(queries)}")
else:
    hit_orgs = {}
    hit_hosts = set()
    print("PHASE 1: No hits. Running Phase 2 across ALL orgs as a broader check.")

### Phase 2 - Deep Dive (hit orgs, Linux only)

Targeted queries against hit orgs for the openssl encryption pipeline and systemd persistence.

In [ ]:
PHASE2_QUERIES = {
    # RSA-OAEP key wrapping -- openssl pkeyutl with OAEP padding
    "behavior_rsa_oaep_wrap": (
        "-336h | plat == linux | NEW_PROCESS | "
        "event/FILE_PATH contains 'openssl' "
        "and event/COMMAND_LINE contains 'pkeyutl' "
        "and event/COMMAND_LINE contains 'oaep'"
    ),

    # AES-CBC file encryption with key-from-file
    "behavior_aes_file_encrypt": (
        "-336h | plat == linux | NEW_PROCESS | "
        "event/FILE_PATH contains 'openssl' "
        "and event/COMMAND_LINE contains '-aes-256-cbc' "
        "and event/COMMAND_LINE contains '-pass file:'"
    ),

    # systemd user service persistence with sysmon name
    "behavior_sysmon_persistence": (
        "-336h | plat == linux | NEW_PROCESS | "
        "event/COMMAND_LINE contains 'systemctl' "
        "and event/COMMAND_LINE contains '--user' "
        "and event/COMMAND_LINE contains 'sysmon'"
    ),
}

# Target only hit orgs from Phase 1, or all orgs if Phase 1 was clean
phase2_target = hit_orgs if hit_orgs else orgs
phase2_events, phase2_errors = run_queries(PHASE2_QUERIES, phase2_target, label="PHASE 2: DEEP DIVE")

### Phase 3 - Correlate

Merge both phases and build a per-host matrix showing which attack stages were observed. Multiple stages on the same host = high confidence.

In [ ]:
# Merge all events from both phases
all_events = phase1_events + phase2_events
all_errors = phase1_errors + phase2_errors

df = pd.DataFrame(all_events)

if not df.empty:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.sort_values('timestamp', ascending=False).reset_index(drop=True)

    # Map query names to attack stages for the correlation matrix
    STAGE_MAP = {
        'behavior_pth_b64_exec': 'pth_trigger',
        'ioc_attacker_dns': 'dns_c2',
        'behavior_rsa_oaep_wrap': 'encryption',
        'behavior_aes_file_encrypt': 'encryption',
        'behavior_sysmon_persistence': 'persistence',
    }

    df['attack_stage'] = df['query_name'].map(STAGE_MAP)

    # Build per-host attack stage matrix
    print("=" * 70)
    print("PER-HOST ATTACK STAGE MATRIX")
    print("=" * 70)

    host_summary = []
    for (hostname, org_name), group in df.groupby(['hostname', 'org_name']):
        stages = set(group['attack_stage'].dropna().unique())
        first_seen = group['timestamp'].min()
        last_seen = group['timestamp'].max()
        sensor_id = group['sensor_id'].iloc[0]

        host_summary.append({
            'hostname': hostname,
            'org_name': org_name,
            'sensor_id': sensor_id,
            'pth_trigger': 'pth_trigger' in stages,
            'encryption': 'encryption' in stages,
            'persistence': 'persistence' in stages,
            'dns_c2': 'dns_c2' in stages,
            'stages_seen': len(stages),
            'total_events': len(group),
            'first_seen': first_seen,
            'last_seen': last_seen,
        })

    df_hosts = pd.DataFrame(host_summary).sort_values('stages_seen', ascending=False).reset_index(drop=True)

    # Display the matrix
    for _, row in df_hosts.iterrows():
        indicators = []
        if row['pth_trigger']:  indicators.append('PTH')
        if row['encryption']:   indicators.append('ENC')
        if row['persistence']:  indicators.append('PERSIST')
        if row['dns_c2']:       indicators.append('DNS')

        confidence = 'CONFIRMED' if row['stages_seen'] >= 3 else 'HIGH' if row['stages_seen'] >= 2 else 'INVESTIGATE'

        print(f"\n  [{confidence}] {row['hostname']} ({row['org_name']})")
        print(f"    Stages: {' + '.join(indicators)} ({row['stages_seen']}/4)")
        print(f"    Events: {row['total_events']}")
        print(f"    Window: {row['first_seen']} to {row['last_seen']}")
        print(f"    Sensor: {row['sensor_id']}")

    # Expand event data for the full dataset
    event_expanded = pd.json_normalize(df['event_data'].apply(json.loads))
    event_expanded.columns = ['evt_' + c for c in event_expanded.columns]
    df_expanded = pd.concat([df.drop(columns=['event_data']), event_expanded], axis=1)

    print(f"\n{'=' * 70}")
    print(f"SUMMARY: {len(df_hosts)} host(s), {len(df)} total events")
    confirmed = len(df_hosts[df_hosts['stages_seen'] >= 3])
    high = len(df_hosts[df_hosts['stages_seen'] == 2])
    investigate = len(df_hosts[df_hosts['stages_seen'] == 1])
    print(f"  CONFIRMED (3+ stages): {confirmed}")
    print(f"  HIGH (2 stages):       {high}")
    print(f"  INVESTIGATE (1 stage): {investigate}")
    print("=" * 70)
else:
    df_expanded = df
    df_hosts = pd.DataFrame()
    print("No hits across either phase. Environment appears clean.")

### Export

Host summary matrix + full expanded event dataset to CSV for triage in DataWrangler.

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

if not df_hosts.empty:
    # Per-host summary matrix
    summary_file = f"litellm_hunt_hosts_{ts}.csv"
    df_hosts.to_csv(summary_file, index=False)
    print(f"Host summary: {summary_file} ({len(df_hosts)} hosts)")

if not df_expanded.empty:
    # Full expanded event dataset for DataWrangler triage
    events_file = f"litellm_hunt_events_{ts}.csv"
    df_expanded.to_csv(events_file, index=False)
    print(f"Full events:  {events_file} ({len(df_expanded)} events, {len(df_expanded.columns)} columns)")

if df_hosts.empty and df_expanded.empty:
    print("Nothing to export -- no hits found")

### Errors

In [ ]:
if all_errors:
    print(f"{len(all_errors)} error(s):")
    for e in all_errors:
        print(f"  [{e['query']}] {e['org']}: {e['error']}")
else:
    print("No errors")

---

### IOCs

**Network:**

| Indicator | Type | Context |
|-----------|------|---------|
| `models.litellm[.]cloud` | Domain | Exfil endpoint (registered 2026-03-23) |
| `checkmarx[.]zone` | Domain | C2 polling |
| `83.142.209[.]11` | IP | C2 (DEMENIN B.V.) |

**File hashes (SHA-256):**

| Hash | File |
|------|------|
| `71e35aef03099cd1f2d6446734273025a163597de93912df321ef118bf135238` | `litellm_init.pth` (v1.82.8) |
| `a0d229be8efcb2f9135e2ad55ba275b76ddcfeb55fa4370e0a522a5bdee0120b` | `proxy_server.py` (malicious) |
| `d2a0d5f564628773b6af7b9c11f6b86531a875bd2d186d7081ab62748a800ebb` | Wheel (v1.82.8) |
| `8395c3268d5c5dbae1c7c6d4bb3c318c752ba4608cfcd90eb97ffb94a910eac2` | Wheel (v1.82.7) |

**Filesystem:**

| Path | What |
|------|------|
| `<site-packages>/litellm_init.pth` (34,628 bytes) | `.pth` auto-exec trigger |
| `~/.config/sysmon/sysmon.py` | C2 backdoor |
| `~/.config/systemd/user/sysmon.service` | Persistence |
| `/tmp/pglog` | Second-stage binary |
| `/tmp/tpcp.tar.gz` | Encrypted exfil archive |

### References

- [HuskyHacks - LiteLLM Credential Stealer Analysis](https://huskyhacks.io/posts/litellm-cred-stealer/)
- [Wiz - TeamPCP Trojanizes LiteLLM](https://www.wiz.io/blog/threes-a-crowd-teampcp-trojanizes-litellm-in-continuation-of-campaign)
- [Datadog - LiteLLM and Telnyx Compromised on PyPI](https://securitylabs.datadoghq.com/articles/litellm-compromised-pypi-teampcp-supply-chain-campaign/)
- [Snyk - Poisoned Scanner Backdooring LiteLLM](https://snyk.io/articles/poisoned-security-scanner-backdooring-litellm/)
- [Sonatype - Multi-Stage Credential Stealer](https://www.sonatype.com/blog/compromised-litellm-pypi-package-delivers-multi-stage-credential-stealer)
- [ramimac - TeamPCP Timeline and IOCs](https://ramimac.me/teampcp/)
- [GitHub Issue #24512 - Original Disclosure](https://github.com/BerriAI/litellm/issues/24512)